In [0]:
dbutils.widgets.removeAll()

In [0]:
# DBTITLE 1, Define Global Pipeline Parameters
dbutils.widgets.text(
    "GoldConfigPath", 
    "/Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing/DimQualityMeasure/Gold/Config/dimQualityMeasure.json", 
    "Target Gold Config JSON Path"
)

# Extract widget inputs to variables
config_path = dbutils.widgets.get("GoldConfigPath")

In [0]:
# DBTITLE 2, Environment and Dependency Setup
import os
import sys
from pathlib import Path

# Force local file system synchronization
os.sync()

# Absolute workspace configuration
ROOT_DIR = Path("/Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing")
sys.path.append(str(ROOT_DIR))

In [0]:
# DBTITLE 3, Gold Layer Execution Function
def trigger_gold_processing(config_file_path: str):
    """Triggers the generalized Gold loop metadata engine."""
    gold_notebook_path = f"{ROOT_DIR}/DimQualityMeasure/Gold/Notebooks/GenericSubGroupProcessing"
    
    print(f"--> Invoking Gold Layer Engine targeting configuration...")
    print(f"    Config Location: {config_file_path}")
    
    payload_arguments = {
        "SubGroupConfigPath": config_file_path
    }
    
    # Run downstream processing engine notebook
    result = dbutils.notebook.run(gold_notebook_path, 1200, arguments=payload_arguments)
    print(f"    Gold Process Response: {result}")
    return result

In [0]:
# DBTITLE 4, Main Pipeline Execution Flow
def main():
    print(f"=======================================================")
    # 2026-07-09 Execution Context
    print(f"STARTING LIFECYCLE FOR PIPELINE: QualityMeasure_Gold")
    print(f"=======================================================")
    
    try:
        # Step 1: Execute Gold Table Processing and Merge
        pipeline_result = trigger_gold_processing(config_path)
        
        print("\n=======================================================")
        print("PIPELINE EXECUTION COMPLETE")
        print(f"Final Outcome: {pipeline_result}")
        print("=======================================================")
        
        dbutils.notebook.exit(f"SUCCESS: {pipeline_result}")
        
    except Exception as e:
        error_msg = f"Pipeline execution faulted during processing: {str(e)}"
        print(f"CRITICAL: {error_msg}")
        dbutils.notebook.exit(f"FAILED: {error_msg}")

In [0]:
if __name__ == "__main__":
    main()